In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from typing import TypedDict, List

class AgentState(TypedDict):
    query: str
    documents: List[str]
    analysis: str
    score: float

In [3]:
def retrieve_data(state: AgentState):
    query = state["query"]
    
    # mock (replace with vector DB / API)
    docs = [
        "Tribal literacy rate is 45%",
        "Average income is below poverty line",
        "Limited access to healthcare"
    ]
    
    return {"documents": docs}

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

def analyze(state: AgentState):
    docs = "\n".join(state["documents"])
    
    prompt = f"""
    Analyze the socio-economic condition based on:
    {docs}
    
    Provide a structured summary.
    """
    
    response = llm.invoke(prompt)
    
    return {"analysis": response.content}

In [5]:
def evaluate(state: AgentState):
    analysis = state["analysis"]
    
    eval_prompt = f"""
    Evaluate the following analysis on:
    1. completeness
    2. factual grounding
    3. clarity
    
    Give a score between 0 and 1.
    
    Analysis:
    {analysis}
    
    Only return a number.
    """
    
    response = llm.invoke(eval_prompt)
    
    try:
        score = float(response.content.strip())
    except:
        score = 0.0
    
    return {"score": score}

In [6]:
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve_data)
builder.add_node("analyze", analyze)
builder.add_node("evaluate", evaluate)

builder.set_entry_point("retrieve")

builder.add_edge("retrieve", "analyze")
builder.add_edge("analyze", "evaluate")
builder.add_edge("evaluate", END)

graph = builder.compile()

In [7]:
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve_data)
builder.add_node("analyze", analyze)
builder.add_node("evaluate", evaluate)

builder.set_entry_point("retrieve")

builder.add_edge("retrieve", "analyze")
builder.add_edge("analyze", "evaluate")
builder.add_edge("evaluate", END)

graph = builder.compile()

print(graph)

In [10]:
result = graph.invoke({
    "query": "Analyze tribal socio-economic conditions"
})

print(result)

{'query': 'Analyze tribal socio-economic conditions', 'documents': ['Tribal literacy rate is 45%', 'Average income is below poverty line', 'Limited access to healthcare'], 'analysis': "### Socio-Economic Condition Analysis\n\n#### Overview\nThe socio-economic condition of the tribal community in question is challenging due to low educational attainment, widespread poverty, and inadequate healthcare facilities. These factors are interconnected and contribute to a cycle of disadvantage impacting the community's overall well-being.\n\n#### Key Indicators\n\n1. **Tribal Literacy Rate: 45%**\n   - **Educational Challenges**: The low literacy rate suggests limited access to quality education and educational facilities. This impacts the community’s ability to access better employment opportunities, further perpetuating the cycle of poverty.\n   - **Cultural Factors**: There may also be cultural or linguistic barriers that hinder educational progress, requiring tailored educational programs.\n